In [1]:
import os
import re
import glob
from docling.document_converter import DocumentConverter
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import chromadb

INPUT_PDF_DIR = "./papers_to_process" # Directory containing your PDF papers
OUTPUT_MD_DIR = "./markdown_outputs"
QUERY = "Identify the experimental section. Extract the specific values for Laser Power (W) and Scanning Speed (mm/s) used for Ti64 fabrication."

# 1. Pre-initialize models (Run once)
print("Initializing models...")
converter = DocumentConverter()
embed_model = SentenceTransformer('BAAI/bge-m3')
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)

def get_or_convert_md(pdf_path):
    """
    Checks if a Markdown version exists locally. 
    If not, converts the PDF and saves the .md file.
    """
    if not os.path.exists(OUTPUT_MD_DIR):
        os.makedirs(OUTPUT_MD_DIR)
        
    filename = os.path.basename(pdf_path).replace(".pdf", ".md")
    md_path = os.path.join(OUTPUT_MD_DIR, filename)
    
    if os.path.exists(md_path):
        print(f"  [Skip] Existing Markdown found: {filename}")
        with open(md_path, "r", encoding="utf-8") as f:
            return f.read(), md_path
    else:
        print(f"  [Convert] Converting PDF: {filename}...")
        result = converter.convert(pdf_path)
        md_content = result.document.export_to_markdown()
        with open(md_path, "w", encoding="utf-8") as f:
            f.write(md_content)
        return md_content, md_path

def process_single_paper(md_content, query):
    """
    Cleans the content, splits it into chunks, 
    and retrieves the top 4 segments relevant to the query.
    """
    # Remove the References/Bibliography section to avoid noise
    body_text = re.split(r'\n#+.*(?:References|Bibliography|参考文献)', md_content, flags=re.IGNORECASE)[0]
    chunks = splitter.split_text(body_text)
    
    # Use an ephemeral (in-memory) database for fast retrieval per paper
    client = chromadb.EphemeralClient()
    
    # Use get_or_create_collection to prevent "Collection already exists" errors
    collection = client.get_or_create_collection(name="temp_paper")
    
    # Encode chunks and add to the vector database
    embeddings = embed_model.encode(chunks).tolist()
    collection.add(
        embeddings=embeddings,
        documents=chunks,
        ids=[f"id_{i}" for i in range(len(chunks))]
    )
    
    # Perform the similarity search
    query_vec = embed_model.encode([query]).tolist()
    results = collection.query(query_embeddings=query_vec, n_results=4)
    
    return "\n\n".join(results['documents'][0])

# --- Main Execution Loop ---
all_pdf_files = glob.glob(os.path.join(INPUT_PDF_DIR, "*.pdf"))
final_prompts = {}

print(f"Starting batch process. Found {len(all_pdf_files)} papers.")

for pdf_path in all_pdf_files:
    paper_name = os.path.basename(pdf_path)
    print(f"\n>>> Processing: {paper_name}")
    
    # Step 1: Retrieve or create Markdown (Caching logic)
    content, md_path = get_or_convert_md(pdf_path)
    
    # Step 2: Extract relevant RAG context
    context = process_single_paper(content, QUERY)
    
    # Step 3: Construct the final Prompt for the LLM
    prompt = f"### CONTEXT FROM {paper_name}\n{context}\n\n### QUERY\n{QUERY}"
    final_prompts[paper_name] = prompt

print("\nAll papers processed successfully!")
print("You can now access the prepared inputs using the 'final_prompts' dictionary.")

/opt/anaconda3/envs/dl/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Initializing models...
Starting batch process. Found 8 papers.

>>> Processing: paper7.pdf
  [Skip] Existing Markdown found: paper7.md

>>> Processing: paper5.pdf
  [Skip] Existing Markdown found: paper5.md

>>> Processing: paper4.pdf
  [Skip] Existing Markdown found: paper4.md

>>> Processing: paper2.pdf
  [Skip] Existing Markdown found: paper2.md

>>> Processing: paper25.pdf
  [Skip] Existing Markdown found: paper25.md

>>> Processing: paper26.pdf
  [Skip] Existing Markdown found: paper26.md

>>> Processing: paper22.pdf
  [Skip] Existing Markdown found: paper22.md

>>> Processing: paper23.pdf
  [Skip] Existing Markdown found: paper23.md

All papers processed successfully!
You can now access the prepared inputs using the 'final_prompts' dictionary.


In [ ]:
import os
import glob
import json
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

DOI_PATTERN = re.compile(r'\b(10\.\d{4,9}/[-._;()/:A-Z0-9]+)\b', re.IGNORECASE)

# 1. Define the extracted Metadata structure using Pydantic
class PaperMetadata(BaseModel):
    title: str = Field(description="The title of the scientific paper.")
    authors: list[str] = Field(description="A list of the paper's authors.")
    journal: str = Field(description="The name of the journal, conference, or publication venue. Return 'Not found' if missing.")
    doi: str = Field(description="The DOI (Digital Object Identifier) of the paper (e.g., 10.1016/j.addma...). Return 'Not found' if missing.")

# 2. LLM
llm = ChatOpenAI(
    model="gpt-3.5-turbo", 
    temperature=0,
    api_key="##YOUR_API"
)

# structured output
structured_llm = llm.with_structured_output(PaperMetadata)

# 3. Prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert researcher specializing in extracting structured metadata from scientific papers converted to Markdown. Carefully extract the requested fields. If any information is definitively missing from the text, output 'Not found'."),
    ("human", "Here is the beginning section of the paper:\n\n{text_chunk}")
])

# combine prompt and LLM
extraction_chain = prompt | structured_llm

# 4. Batch processing of Markdown files
def process_markdown_files(directory_path):
    md_files = glob.glob(os.path.join(directory_path, "*.md"))
    if not md_files:
        print(f"No markdown files found in '{directory_path}'.")
        return

    results = {}

    for file_path in md_files:
        filename = os.path.basename(file_path)
        print(f"Processing: {filename}...")
        
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                # Optimization 1: Expand the reading range to 8000 characters, covering the entire first page and part of the second page.
                content = f.read()[:8000] 
            
            # LLM extraction
            metadata = extraction_chain.invoke({"text_chunk": content})
            metadata_dict = metadata.model_dump()
            
            # Optimization 2: Regex Fallback Strategy
            # If the LLM fails to find the DOI, we will scan the truncated text using regular expressions.
            if metadata_dict['doi'].lower() == 'not found':
                match = DOI_PATTERN.search(content)
                if match:
                    metadata_dict['doi'] = match.group(1)
            
            results[filename] = metadata_dict
            print(f"  [Success] Title: {metadata_dict['title']}")
            print(f"  [Success] DOI: {metadata_dict['doi']}")
            
        except Exception as e:
            print(f"  [Error] Failed to process {filename}: {e}")

    output_file = "extracted_metadata.json"
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=4, ensure_ascii=False)
    
    print(f"\nExtraction complete. Results saved to {output_file}")

if __name__ == "__main__":
    # Specify the path of folder
    TARGET_DIR = "markdown_outputs"
    process_markdown_files(TARGET_DIR)

/opt/anaconda3/envs/dl/lib/python3.11/site-packages/langchain_openai/chat_models/base.py:2210: UserWarning: Cannot use method='json_schema' with model gpt-3.5-turbo since it doesn't support OpenAI's Structured Output API. You can see supported models here: https://platform.openai.com/docs/guides/structured-outputs#supported-models. To fix this warning, set `method='function_calling'. Overriding to method='function_calling'.
  warnings.warn(


Processing: paper25.md...
  [Success] Title: Chemical polishing of samples obtained by selective laser melting from titanium alloy Ti6Al4V
  [Success] DOI: Not found
Processing: paper5.md...
  [Success] Title: Critical evaluation of the pulsed selective laser melting process when fabricating Ti64 parts using a range of particle size distributions
  [Success] DOI: Not found
Processing: paper4.md...
  [Success] Title: Residual stress evaluation in selective-laser-melting additively manufactured titanium (Ti-6Al-4V) and inconel 718 using the contour method and numerical simulation
  [Success] DOI: Not found
Processing: paper7.md...
  [Success] Title: Investigating the Residual Stress Distribution in Selective Laser Melting Produced Ti-6Al-4V using Neutron Diffraction
  [Success] DOI: Not found
Processing: paper2.md...
  [Success] Title: Multi-objective accelerated process optimization of mechanical properties in laser-based additive manufacturing: Case study on Selective Laser Melting (SL